In [1]:
print("PASSWORD SECURITY & AUTHENTICATION SYSTEM")
print("-----------------------------------------")
print("Project started successfully!")
print("We will build a secure authentication system.")

PASSWORD SECURITY & AUTHENTICATION SYSTEM
-----------------------------------------
Project started successfully!
We will build a secure authentication system.


In [2]:
import sqlite3
import hashlib
import secrets
import re
import logging
from datetime import datetime, timedelta

from werkzeug.security import generate_password_hash, check_password_hash

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [3]:
import sqlite3

# Connect to the database
connection = sqlite3.connect("password_security.db")

# Create a database cursor
cursor = connection.cursor()

# Create users table
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password_hash TEXT NOT NULL,
    failed_attempts INTEGER DEFAULT 0,
    lockout_until TEXT
)
""")

# Save changes
connection.commit()

print("Database created successfully!")
print("Users table created successfully!")

# Close connection
connection.close()

Database created successfully!
Users table created successfully!


In [4]:
def check_password_strength(password):

    score = 0

    if len(password) >= 8:
        score += 1

    if re.search(r"[A-Z]", password):
        score += 1

    if re.search(r"[a-z]", password):
        score += 1

    if re.search(r"[0-9]", password):
        score += 1

    if re.search(r"[^A-Za-z0-9]", password):
        score += 1

    if score <= 2:
        return "Weak"

    elif score <= 4:
        return "Medium"

    else:
        return "Strong"


# Test the password checker
test_password = "Hello@123"

result = check_password_strength(test_password)

print("Password:", test_password)
print("Password Strength:", result)

Password: Hello@123
Password Strength: Strong


In [5]:
from werkzeug.security import generate_password_hash, check_password_hash

# Original password
password = "Hello@123"

# Convert password into a secure hash
hashed_password = generate_password_hash(password)

# Display results
print("Original Password:", password)
print("Hashed Password:", hashed_password)

# Verify password
if check_password_hash(hashed_password, password):
    print("Password verification successful!")
else:
    print("Password verification failed!")

Original Password: Hello@123
Hashed Password: scrypt:32768:8:1$uYkT3AtWQkx8woGE$7202f68dfe3c30eed853d35f9deb74b2a808209cd37d24488a7cd9b2a6ec535763fb388d963399294212f2b12f6502a043b2368311d9b7093364c21a5787fd50
Password verification successful!


In [6]:
def register_user(username, password):

    # Check username
    if not username.strip():
        print("Error: Username cannot be empty.")
        return

    # Check minimum password length
    if len(password) < 8:
        print("Error: Password must contain at least 8 characters.")
        return

    # Check password strength
    strength = check_password_strength(password)

    if strength == "Weak":
        print("Error: Please choose a stronger password.")
        return

    # Hash password
    password_hash = generate_password_hash(password)

    # Save user in database
    connection = sqlite3.connect("password_security.db")
    cursor = connection.cursor()

    try:
        cursor.execute(
            """INSERT INTO users (username, password_hash)
               VALUES (?, ?)""",
            (username, password_hash)
        )

        connection.commit()
        print("Registration successful!")
        print("Password Strength:", strength)

    except sqlite3.IntegrityError:
        print("Error: Username already exists.")

    finally:
        connection.close()


# Test registration with a synthetic account
register_user("student01", "Hello@123")

Registration successful!
Password Strength: Strong


In [7]:
def login_user(username, password):

    connection = sqlite3.connect("password_security.db")
    cursor = connection.cursor()

    cursor.execute(
        "SELECT password_hash FROM users WHERE username = ?",
        (username,)
    )

    user = cursor.fetchone()
    connection.close()

    if user is None:
        print("Login failed: Invalid username or password.")
        return False

    stored_hash = user[0]

    if check_password_hash(stored_hash, password):
        print("Login successful! Welcome", username)
        return True

    else:
        print("Login failed: Invalid username or password.")
        return False


# Test login
login_user("student01", "Hello@123")

Login successful! Welcome student01


True

In [8]:
login_user("student01", "WrongPassword123")

Login failed: Invalid username or password.


False

In [13]:
def login_user_with_lockout(username, password):

    connection = sqlite3.connect("password_security.db")
    cursor = connection.cursor()

    cursor.execute(
        """SELECT password_hash, failed_attempts, lockout_until
           FROM users WHERE username = ?""",
        (username,)
    )

    user = cursor.fetchone()

    if user is None:
        connection.close()
        print("Login failed: Invalid username or password.")
        return False

    stored_hash, failed_attempts, lockout_until = user

    # Check whether account is locked
    if lockout_until:
        unlock_time = datetime.fromisoformat(lockout_until)

        if datetime.now() < unlock_time:
            connection.close()
            print("Account is temporarily locked.")
            return False

        else:
            cursor.execute(
                """UPDATE users
                   SET failed_attempts = 0, lockout_until = NULL
                   WHERE username = ?""",
                (username,)
            )
            failed_attempts = 0

    # Verify password
    if check_password_hash(stored_hash, password):

        cursor.execute(
            """UPDATE users
               SET failed_attempts = 0, lockout_until = NULL
               WHERE username = ?""",
            (username,)
        )

        connection.commit()
        connection.close()

        print("Login successful!")
        return True

    else:
        failed_attempts += 1

        lockout_time = None

        if failed_attempts >= 5:
            lockout_time = (
                datetime.now() + timedelta(minutes=5)
            ).isoformat()

        cursor.execute(
            """UPDATE users
               SET failed_attempts = ?, lockout_until = ?
               WHERE username = ?""",
            (failed_attempts, lockout_time, username)
        )

        connection.commit()
        connection.close()

        if lockout_time:
            print("Account locked for 5 minutes.")
        else:
            print("Incorrect password.")
            print("Failed attempts:", failed_attempts)

        return False

In [14]:
register_user("locktest01", "Secure@123")

Registration successful!
Password Strength: Strong


In [15]:
for attempt in range(1, 6):
    print("Attempt:", attempt)
    login_user_with_lockout("locktest01", "WrongPass123")

Attempt: 1
Incorrect password.
Failed attempts: 1
Attempt: 2
Incorrect password.
Failed attempts: 2
Attempt: 3
Incorrect password.
Failed attempts: 3
Attempt: 4
Incorrect password.
Failed attempts: 4
Attempt: 5
Account locked for 5 minutes.


In [16]:
login_user_with_lockout("locktest01", "Secure@123")

Account is temporarily locked.


False

In [17]:
import logging

# Configure security logging
logging.basicConfig(
    filename="security.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

def log_security_event(event, username):
    logging.info("%s - User: %s", event, username)

# Test security logging
log_security_event("Successful login", "student01")
log_security_event("Failed login", "student01")
log_security_event("Account locked", "locktest01")

print("Security logging configured successfully!")

Security logging configured successfully!


In [18]:
with open("security.log", "r") as file:
    print(file.read())

2026-11-01 14:55:59,116 - INFO - Successful login - User: student01
2026-11-01 14:55:59,116 - INFO - Failed login - User: student01
2026-11-01 14:55:59,117 - INFO - Account locked - User: locktest01



In [19]:
security_recommendations = [
    "Always store hashed passwords, never plaintext passwords.",
    "Use secure password hashing algorithms such as scrypt or Argon2.",
    "Require passwords with a minimum length of 8 characters.",
    "Track repeated failed login attempts.",
    "Temporarily lock accounts after repeated failures.",
    "Never expose passwords in security logs.",
    "Use HTTPS to protect login information.",
    "Use secure session management.",
    "Use synthetic accounts for security testing.",
    "Keep software and security libraries updated."
]

print("PASSWORD SECURITY RECOMMENDATIONS")
print("--------------------------------")

for number, recommendation in enumerate(
    security_recommendations, start=1
):
    print(f"{number}. {recommendation}")

PASSWORD SECURITY RECOMMENDATIONS
--------------------------------
1. Always store hashed passwords, never plaintext passwords.
2. Use secure password hashing algorithms such as scrypt or Argon2.
3. Require passwords with a minimum length of 8 characters.
4. Track repeated failed login attempts.
5. Temporarily lock accounts after repeated failures.
6. Never expose passwords in security logs.
7. Use HTTPS to protect login information.
8. Use secure session management.
9. Use synthetic accounts for security testing.
10. Keep software and security libraries updated.
